# Data Cleaning

In [551]:
import pandas as pd
import numpy as np

df = pd.read_csv("Dataset.csv")

In [552]:
#checking the shape of the dataset
print("Shape of the dataset:", df.shape)
df.head()

Shape of the dataset: (3900, 18)


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [553]:
#listing the columns of the dataset
df.columns.tolist()

['Customer ID',
 'Age',
 'Gender',
 'Item Purchased',
 'Category',
 'Purchase Amount (USD)',
 'Location',
 'Size',
 'Color',
 'Season',
 'Review Rating',
 'Subscription Status',
 'Shipping Type',
 'Discount Applied',
 'Promo Code Used',
 'Previous Purchases',
 'Payment Method',
 'Frequency of Purchases']

In [554]:
#checking the information of the dataset such as data types, non-null values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

In [555]:
#Checking for missing/null values in the dataset
print("Number of null values in each column:")
print(df.isnull().sum())

Number of null values in each column:
Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64


In [556]:
missing_percentage = df['Review Rating'].isnull().sum() / len(df) * 100
print("Percentage of missing values in Review Rating Column:")
print(missing_percentage)

Percentage of missing values in Review Rating Column:
0.9487179487179488


In [557]:
print("Total Rows      :", len(df))
print("Unique Customers:", df["Customer ID"].nunique())
print("Duplicate IDs   :", df["Customer ID"].duplicated().sum())

Total Rows      : 3900
Unique Customers: 3900
Duplicate IDs   : 0


### Key Findings

- Dataset contains 3900 customer records.
- Customer ID is unique for every row.
- Only Review Rating contains missing values (37 records, 0.95%).
- No duplicate customers found.
- Dataset is customer-level, not transaction-level.
- No timestamps or churn labels are available.

In [558]:
df["Review Rating"].isnull().sum()
df["Review Rating"] = (
    df["Review Rating"]
    .fillna(df["Review Rating"].median())
)

print(df.isnull().sum())

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64


### Observation

Only 37 ratings are missing (0.95% of the dataset).

### Decision

Median imputation is appropriate because:

- Missingness is very small.
- No significant pattern is visible.
- Removing rows would unnecessarily reduce sample size.

In [559]:
pd.crosstab(
    df["Discount Applied"],
    df["Promo Code Used"]
)


Promo Code Used,No,Yes
Discount Applied,,
No,2223,0
Yes,0,1677


In [560]:

df.drop(columns=["Promo Code Used"], inplace=True)

### Observation

Discount Applied and Promo Code Used are perfectly aligned.

Every customer who used a promo code received a discount, and every
customer who received a discount used a promo code.

### Decision

'Promo Code Used' is redundant and can be dropped to avoid duplicate
information during feature engineering.

In [561]:
pd.crosstab(
    df["Subscription Status"],
    df["Discount Applied"]
)

Discount Applied,No,Yes
Subscription Status,,
No,2223,624
Yes,0,1053


### Observation

- All subscribers received discounts.
- 624 non-subscribers also received discounts.
- 2223 non-subscribers paid full price.

### Insight

Subscription and discount usage are strongly related but not identical.

Subscribers form a subset of discount recipients.

This suggests the subscription program is partly a discount mechanism,
but discounts are also distributed through other promotional channels.

In [562]:
#checking the frequency of purchases made by customers
df["Frequency of Purchases"].value_counts()

Frequency of Purchases
Every 3 Months    584
Annually          572
Quarterly         563
Monthly           553
Bi-Weekly         547
Fortnightly       542
Weekly            539
Name: count, dtype: int64

### Observation

Two pairs of labels represent the same purchase cadence:

- Fortnightly = Bi-Weekly
- Quarterly = Every 3 Months

### Decision

These labels will be consolidated before converting frequency into a
numeric score.

In [563]:
print(
    "Minimum:",
    df["Purchase Amount (USD)"].min()
)

print(
    "Maximum:",
    df["Purchase Amount (USD)"].max()
)

print(
    "Integer Values Only:",
    (df["Purchase Amount (USD)"] % 1 == 0).all()
)

Minimum: 20
Maximum: 100
Integer Values Only: True


### Observation

Purchase amounts range from $20 to $100 and contain only integer values.

### Insight

The narrow and highly controlled range suggests a synthetic dataset
with predefined limits rather than real-world transaction data.

In [564]:
print(
    "Minimum:",
    df["Previous Purchases"].min()
)

print(
    "Maximum:",
    df["Previous Purchases"].max()
)

Minimum: 1
Maximum: 50


### Observation

Previous Purchases ranges from 1 to 50.

### Insight

There are no customers with zero previous purchases.

Therefore every row represents a returning customer rather than a
first-time buyer.

In [565]:
df["Purchase Amount (USD)"].corr(
    df["Previous Purchases"]
)

np.float64(0.008063412270587713)

### Observation

The correlation between Purchase Amount and Previous Purchases is close
to zero.

### Insight

Customers who have purchased many times do not necessarily spend more
per transaction.

This indicates that Purchase Amount should not be treated as a proxy
for customer lifetime value on its own.

A composite value metric will be required during feature engineering.

In [566]:
#here we are converting the categorical values of "Discount Applied" and "Subscription Status" columns into numerical values for further analysis. 
#The "Yes" values will be converted to 1 and "No" values will be converted to 0.

print(df["Discount Applied"].dtype)
print(df["Discount Applied"].unique())

df["Discount Applied"] = (
    df["Discount Applied"] == "Yes"
).astype(int)

df["Subscription Status"] = (
    df["Subscription Status"] == "Yes"
).astype(int)

str
<StringArray>
['Yes', 'No']
Length: 2, dtype: str


# Feature Engineering

## Feature 1: Promo Persona

Purpose:
Classify customers based on their relationship with promotions.

Segments:
- Subscriber → Subscriber receiving discounts
- Promo Opportunist → Non-subscriber receiving discounts
- Organic → Non-subscriber paying full price

Business Use:
Used later for loyalty analysis and promotion strategy.

In [567]:
conditions = [
    (df["Subscription Status"] == 1) &
    (df["Discount Applied"] == 1),

    (df["Subscription Status"] == 0) &
    (df["Discount Applied"] == 1),

    (df["Subscription Status"] == 0) &
    (df["Discount Applied"] == 0)
]

choices = [
    "Subscriber",
    "Promo_Opportunist",
    "Organic"
]

df["promo_persona"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

df["promo_persona"].value_counts()

promo_persona
Organic              2223
Subscriber           1053
Promo_Opportunist     624
Name: count, dtype: int64

## Feature 2: Purchase Frequency Score

Purpose:
Convert purchase frequency labels into a numerical scale.

Why?
Weekly customers are much more engaged than annual customers.

This allows frequency to be used in scoring and segmentation.

In [568]:
freq_map = {
    "Weekly": 52,
    "Bi-Weekly": 26,
    "Fortnightly": 26,
    "Monthly": 12,
    "Quarterly": 4,
    "Every 3 Months": 4,
    "Annually": 1
}

df["purchase_frequency_score"] = (
    df["Frequency of Purchases"]
    .map(freq_map)
)

df["purchase_frequency_score"].value_counts().sort_index()

purchase_frequency_score
1      572
4     1147
12     553
26    1089
52     539
Name: count, dtype: int64

## Feature 3: Tenure Tier

Purpose:
Create customer lifecycle segments using Previous Purchases.

Segments:
- New
- Growing
- Established
- Loyal

In [569]:
def tenure_tier(x):

    if x <= 13:
        return "New"

    elif x <= 25:
        return "Growing"

    elif x <= 38:
        return "Established"

    else:
        return "Loyal"

df["tenure_tier"] = (
    df["Previous Purchases"]
    .apply(tenure_tier)
)

df["tenure_tier"].value_counts()

tenure_tier
New            1014
Established    1007
Growing         951
Loyal           928
Name: count, dtype: int64

## Feature 4: Promo Dependency Score

Purpose:
Estimate how dependent a customer is on discounts.

Logic:
Customers with low tenure and low purchase frequency are assumed
to be more promotion dependent.

Customers with high tenure and high frequency are assumed to be
less dependent on promotions.

In [570]:
freq_norm = (
    (df["purchase_frequency_score"] - 1)
    / (52 - 1)
)

tenure_norm = (
    (df["Previous Purchases"] - 1)
    / (50 - 1)
)

df["promo_dependency_score"] = (
    df["Discount Applied"]
    *
    (
        1
        - 0.5 * freq_norm
        - 0.5 * tenure_norm
    )
).clip(0, 1)

df["promo_dependency_score"].describe()

count    3900.000000
mean        0.251642
std         0.324973
min         0.000000
25%         0.000000
50%         0.000000
75%         0.550820
max         1.000000
Name: promo_dependency_score, dtype: float64

In [571]:
df.groupby("promo_persona")[
    "promo_dependency_score"
].mean().round(3)

promo_persona
Organic              0.000
Promo_Opportunist    0.598
Subscriber           0.578
Name: promo_dependency_score, dtype: float64

## Feature 5: Value Tier

Purpose:
Identify the brand's most valuable customers.

Value Proxy:
Purchase Amount × Previous Purchases × Purchase Frequency

This captures:
- How much they spend
- How long they have been buying
- How frequently they buy

In [572]:
raw_value = (
    df["Purchase Amount (USD)"]
    *
    df["Previous Purchases"]
    *
    df["purchase_frequency_score"]
)

value_pct = raw_value.rank(pct=True)

df["value_tier"] = pd.cut(
    value_pct,
    bins=[0, 0.25, 0.50, 0.75, 1.00],
    labels=["Low", "Medium", "High", "Premium"],
    include_lowest=True
)

df["value_tier"].value_counts()

value_tier
Low        976
Premium    976
Medium     974
High       974
Name: count, dtype: int64

## Feature 6: Satisfaction Flag

Purpose:
Identify satisfied customers.

Satisfied:
Rating ≥ 4

Not Satisfied:
Rating < 4

In [573]:
df["satisfaction_flag"] = (
    df["Review Rating"] >= 4
).astype(int)

df["satisfaction_flag"].value_counts()

satisfaction_flag
0    2266
1    1634
Name: count, dtype: int64

## Feature 7: Age Group

Purpose:
Create demographic segments for customer profiling.

In [574]:
def age_group(age):

    if age <= 25:
        return "18-25"

    elif age <= 35:
        return "26-35"

    elif age <= 50:
        return "36-50"

    else:
        return "51-70"

df["age_group"] = (
    df["Age"]
    .apply(age_group)
)

df["age_group"].value_counts()

age_group
51-70    1476
36-50    1111
26-35     742
18-25     571
Name: count, dtype: int64

## Feature 8: Premium Shipping Flag

Purpose:
Identify customers choosing faster shipping options.

Hypothesis:
Customers willing to pay for premium shipping may be less
price-sensitive.

In [575]:
premium_shipping = [
    "Next Day Air",
    "Express",
    "2-Day Shipping"
]

df["is_premium_shipping"] = (
    df["Shipping Type"]
    .isin(premium_shipping)
).astype(int)

df["is_premium_shipping"].value_counts()

is_premium_shipping
0    1979
1    1921
Name: count, dtype: int64

## Feature 9: Digital Payer

Purpose:
Identify customers who use digital payment methods.

These customers may be easier to target through digital campaigns.

In [576]:
df["is_digital_payer"] = (
    df["Payment Method"] != "Cash"
).astype(int)

df["is_digital_payer"].value_counts()

is_digital_payer
1    3230
0     670
Name: count, dtype: int64

## Feature 10: Loyalty Composite Score

Purpose:
Create a loyalty measure using:

- Purchase frequency
- Customer tenure
- Promo dependency
- Satisfaction

Higher score = stronger loyalty.

In [577]:
freq_norm = (
    (df["purchase_frequency_score"] - 1)
    / (52 - 1)
)

tenure_norm = (
    (df["Previous Purchases"] - 1)
    / (50 - 1)
)

df["loyalty_score"] = (
    freq_norm
    *
    tenure_norm
    *
    (1 - df["promo_dependency_score"])
    *
    (1 + 0.2 * df["satisfaction_flag"])
)

df["loyalty_score"].describe()

count    3900.000000
mean        0.149465
std         0.223311
min         0.000000
25%         0.006629
50%         0.047188
75%         0.204082
max         1.200000
Name: loyalty_score, dtype: float64

In [578]:
df.groupby("promo_persona")[
    "loyalty_score"
].mean().round(3)

promo_persona
Organic              0.170
Promo_Opportunist    0.112
Subscriber           0.128
Name: loyalty_score, dtype: float64

In [579]:
df.columns.tolist()

['Customer ID',
 'Age',
 'Gender',
 'Item Purchased',
 'Category',
 'Purchase Amount (USD)',
 'Location',
 'Size',
 'Color',
 'Season',
 'Review Rating',
 'Subscription Status',
 'Shipping Type',
 'Discount Applied',
 'Previous Purchases',
 'Payment Method',
 'Frequency of Purchases',
 'promo_persona',
 'purchase_frequency_score',
 'tenure_tier',
 'promo_dependency_score',
 'value_tier',
 'satisfaction_flag',
 'age_group',
 'is_premium_shipping',
 'is_digital_payer',
 'loyalty_score']

In [580]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

Rows : 3900
Columns : 27


In [581]:
engineered_features = [
    "promo_persona",
    "purchase_frequency_score",
    "tenure_tier",
    "promo_dependency_score",
    "value_tier",
    "satisfaction_flag",
    "age_group",
    "is_premium_shipping",
    "is_digital_payer",
    "loyalty_score"
]

pd.DataFrame({
    "Feature": engineered_features
})

,Feature
0,promo_persona
1,purchase_frequency_score
2,tenure_tier
3,promo_dependency_score
4,value_tier
5,satisfaction_flag
6,age_group
7,is_premium_shipping
8,is_digital_payer
9,loyalty_score


## Phase 1 Summary

### Data Cleaning

- Removed redundant column: Promo Code Used
- Imputed missing Review Ratings using median
- Standardized purchase frequency categories
- Binary encoded subscription and discount indicators

### Engineered Features

1. promo_persona
2. purchase_frequency_score
3. tenure_tier
4. promo_dependency_score
5. value_tier
6. satisfaction_flag
7. age_group
8. is_premium_shipping
9. is_digital_payer
10. loyalty_score

### Final Dataset

- Original Columns: 18
- Dropped Columns: 1
- Engineered Features: 10
- Final Columns: 27
- Total Customers: 3900

In [582]:
df.to_csv(
    "engineered_dataset.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


In [583]:
df.isnull().sum()

Customer ID                 0
Age                         0
Gender                      0
Item Purchased              0
Category                    0
Purchase Amount (USD)       0
Location                    0
Size                        0
Color                       0
Season                      0
Review Rating               0
Subscription Status         0
Shipping Type               0
Discount Applied            0
Previous Purchases          0
Payment Method              0
Frequency of Purchases      0
promo_persona               0
purchase_frequency_score    0
tenure_tier                 0
promo_dependency_score      0
value_tier                  0
satisfaction_flag           0
age_group                   0
is_premium_shipping         0
is_digital_payer            0
loyalty_score               0
dtype: int64